# 08.5 — Weights & Biases integration

MATLAB's pipeline has a bespoke "monitor" — a live training plot and run bookkeeping ([08.3](08.3_monitor_table_compatibility.ipynb)). The modern Python equivalent is **Weights & Biases (W&B)**: a hosted experiment tracker that logs metrics live, draws dashboards, compares runs, and orchestrates hyperparameter sweeps. This project *declares* W&B as a dependency and *names* it as the monitor's replacement — but here's the honest part: **there is no W&B code in the project yet.** It's a planned integration with a clean extension point already in place. This notebook teaches W&B, shows the honest state, and demonstrates exactly how you'd wire it in.

This notebook covers:

* What W&B does and why it replaces a bespoke monitor.
* The honest state: declared dependency, no code (aspirational).
* The W&B API (`init` / `log` / `finish`), run live in disabled mode.
* Wiring W&B into the existing `EpochCallback` hook.

**Prerequisites:** [08.3 monitor table compatibility](08.3_monitor_table_compatibility.ipynb), [00.7 pip & packaging](../00_orientation/00.7_pip_packaging_and_project_anatomy.ipynb).

## Section 1 — What MATLAB does

MATLAB's `trainingProgressMonitor` (or the older `'training-progress'` plot) shows live loss/accuracy curves during training and tracks the run's metadata:

```matlab
monitor = trainingProgressMonitor(Metrics=["Loss","Accuracy"], XLabel="Epoch");
for epoch = 1:numEpochs
    recordMetrics(monitor, epoch, Loss=loss, Accuracy=acc);   % live plot updates
end
```

It's a *live display* plus *run bookkeeping* — but it's bespoke, local, and doesn't compare runs or manage sweeps. The migration plan's choice: replace it with **W&B for live telemetry**, keeping the `.mat` dump ([08.2](08.2_writing_mat_files_for_matlab.ipynb)) for the MATLAB analysis bridge. W&B is the industry-standard version of what the monitor did, and far more.

## Section 2 — The Python concepts you need

### 2.1 — What W&B is

Weights & Biases is an **experiment tracker**. During a run you `log` metrics (loss, accuracy, learning rate) each step; W&B streams them to a dashboard with live plots. Across runs it lets you *compare* configurations side by side, *group* runs, sweep hyperparameters, and version model artifacts. It's the modern replacement for both a bespoke monitor *and* a spreadsheet of results — the standard tool for tracking deep-learning experiments. The core loop is three calls: `wandb.init()` to start a run, `run.log({...})` per step, `run.finish()` at the end.

### 2.2 — The honest state: declared, not wired

In [ ]:
from pathlib import Path

# W&B IS a declared dependency:
pyproject = Path("../../pyproject.toml").read_text().splitlines()
for line in pyproject:
    if "wandb" in line:
        print("pyproject.toml:", line.strip())

# ...but there is NO wandb code in the source tree:
src_files = list(Path("../../src/neural_data_decoding").rglob("*.py"))
uses_wandb = [f for f in src_files if "import wandb" in f.read_text() or "wandb." in f.read_text()]
print(f"\nsource files that import/use wandb: {len(uses_wandb)}")
print("→ declared dependency, zero usage: W&B is aspirational — planned, with a hook ready (§2.4).")

This is the third "registered ≠ implemented" honesty point in the curriculum ([03.5](../03_data_pipeline/03.5_normalization_strategies.ipynb) normalization recipes, [08.3](08.3_monitor_table_compatibility.ipynb) the monitor stub, and now W&B). The dependency is in `pyproject.toml` and the docs name W&B as the monitor's successor, but no code calls it. Today's telemetry is the stdout epoch print ([08.3 §2.5](08.3_monitor_table_compatibility.ipynb)). That's not a criticism — it's a common, sensible state for a research port: get the *numerics* right first, add *observability* when you start running real sweeps. The important thing is that the *hook* to add it already exists (§2.4).

### 2.3 — The W&B API, run live

In [ ]:
import os
os.environ["WANDB_SILENT"] = "true"
os.environ["WANDB_MODE"] = "disabled"      # no network/auth — safe for notebooks & CI
import wandb

# The three-call core loop: init → log per step → finish.
run = wandb.init(project="neural-decoding-demo", config={"lr": 1e-3, "model": "GRU"})
val_accs = [0.42, 0.55, 0.61, 0.63]
for epoch, acc in enumerate(val_accs, start=1):
    run.log({"epoch": epoch, "val_accuracy": acc})     # streams to the dashboard (when online)
run.summary["best_val_accuracy"] = max(val_accs)        # a run-level summary metric
run.finish()
print(f"\nlogged {len(val_accs)} epochs; best val accuracy {max(val_accs)} recorded as a summary metric.")
print("(mode='disabled' → no server; online, these become live plots and a comparable run.)")

In `disabled` mode W&B no-ops the network calls, so this runs anywhere (notebooks, CI) without an account. Online (`mode="online"` with an API key), the same three calls produce a live dashboard: `val_accuracy` becomes a plot updating each epoch, `config` tags the run for filtering, and `summary["best_val_accuracy"]` is the headline number you sort runs by. The API is deliberately tiny — `init`, `log`, `finish` — so wiring it into an existing training loop is a few lines.

### 2.4 — Wiring W&B into the `EpochCallback` hook

In [ ]:
# The lifecycle already exposes an EpochCallback fired every epoch (08.3 §2.3).
# A W&B logger is just a callback that logs the epoch's metrics — no core changes.
def make_wandb_callback(run):
    def on_epoch(epoch, train_loss, val_acc, is_best):
        run.log({"epoch": epoch, "train_loss": train_loss,
                 "val_accuracy": val_acc, "is_optimal": int(is_best)})
    return on_epoch

run = wandb.init(project="neural-decoding-demo", mode="disabled")
cb = make_wandb_callback(run)
# Simulate the lifecycle calling the callback each epoch:
for e, (tl, va, best) in enumerate([(0.9, 0.5, True), (0.6, 0.58, True), (0.5, 0.57, False)], start=1):
    cb(e, tl, va, best)
    print(f"epoch {e}: logged train_loss={tl}, val_acc={va}, optimal={best}")
run.finish()
print("→ the callback IS the integration point — pass this to fit_supervised(epoch_callback=...).")

No core code changes needed: the training lifecycle already fires an `EpochCallback` every epoch ([08.3 §2.3](08.3_monitor_table_compatibility.ipynb)), so a W&B logger is *just a callback* you pass in. That's why the absent integration (§2.2) is a small gap, not a redesign — the extension point is deliberately there. The same hook could drive TensorBoard, MLflow, a CSV logger, or all at once.

### 2.5 — W&B and the .mat table are complementary

A crucial distinction: W&B and the `CM_Table.mat` ([08.2](08.2_writing_mat_files_for_matlab.ipynb)) serve *different* purposes and you keep *both*:

| | Weights & Biases | `CM_Table.mat` |
|---|---|---|
| audience | the human watching the run (live) | the MATLAB analysis scripts (durable) |
| when | streamed every epoch | written on new-best + at the end |
| purpose | observability, run comparison, sweeps | MATLAB-compatible final results |

The plan's phrasing is "W&B **plus** a minimal `.mat` dump for the MATLAB bridge." W&B is your live cockpit; the `.mat` file is the deliverable the downstream MATLAB pipeline consumes ([08.6](08.6_running_matlab_analysis_on_python_output.ipynb)). Adding W&B doesn't replace the `.mat` writer — they answer different questions.

## Section 3 — The `neural_data_decoding` implementation

The dependency declaration and the hook it would attach to:

In [ ]:
from pathlib import Path
# The declared dependency (Telemetry section of pyproject.toml):
pp = Path("../../pyproject.toml").read_text().splitlines()
i = next(j for j, line in enumerate(pp) if "wandb" in line)
for k in range(max(0, i - 1), i + 1):
    print(f"pyproject.toml {k + 1:3} | {pp[k].strip()}")
print()
# The hook a W&B logger attaches to:
lc = Path("../../src/neural_data_decoding/training/lifecycle.py").read_text().splitlines()
j0 = next(j for j, line in enumerate(lc) if "EpochCallback = " in line)
print(f"lifecycle.py {j0 + 1:3} | {lc[j0].strip()}   # ← attach a W&B logger here")

Things to spot:

* `wandb>=0.16` sits under the Telemetry section of `pyproject.toml` — installed, importable ([08.2](08.2_writing_mat_files_for_matlab.ipynb)'s `scipy` is in the same install), but unused by `src/`.
* `EpochCallback = Callable[[EpochHistory], None]` is the attachment point — `fit_supervised(..., epoch_callback=your_wandb_logger)`. The `monitoring/` stub docstring ([08.3 §2.1](08.3_monitor_table_compatibility.ipynb)) even names "W&B logger" as its first planned occupant.
* The `disabled` mode (`WANDB_MODE=disabled`) is how you keep W&B calls in the code without breaking CI or offline runs — the logger no-ops when there's no account. A robust integration guards `wandb.init` so a missing API key downgrades gracefully.
* When you build it: log per-epoch metrics via the `EpochCallback`, and the run-level "best" via the `OnOptimalCallback` ([08.3](08.3_monitor_table_compatibility.ipynb)) — the two hooks map cleanly onto W&B's `log` (per step) and `summary` (per run).

## Section 4 — Hands-on exercises

### Exercise 4.1 — log a config sweep

Run three disabled-mode W&B runs with different learning rates and record each one's best accuracy in its summary — the pattern behind a hyperparameter sweep.

In [ ]:
# Reveal:
import math
results = {}
for lr in (1e-2, 1e-3, 1e-4):
    run = wandb.init(project="sweep-demo", config={"lr": lr}, mode="disabled", reinit=True)
    best = 0.5 + 0.1 * -abs(math.log10(lr) + 3)     # toy: lr=1e-3 is best
    run.summary["best_val_accuracy"] = best
    results[lr] = best
    run.finish()
for lr, acc in results.items():
    print(f"lr={lr}: best val accuracy {acc:.3f}")
print("→ online, W&B's sweep view sorts these automatically; here we just collect the summaries.")

### Exercise 4.2 — the callback is logger-agnostic

Write the *same* epoch callback to log to a plain list instead of W&B, showing the hook doesn't care what backend you use.

In [ ]:
# Reveal:
csv_log = []
def csv_callback(epoch, train_loss, val_acc, is_best):
    csv_log.append((epoch, train_loss, val_acc, is_best))

for e, (tl, va, best) in enumerate([(0.9, 0.5, True), (0.6, 0.58, True)], start=1):
    csv_callback(e, tl, va, best)
print("logged rows:", csv_log)
print("→ W&B, CSV, TensorBoard — all the same EpochCallback shape. The hook is backend-agnostic.")

## Section 5 — Common errors

### `wandb.init` hangs or asks for a login

It's trying to reach the server. Use `mode="disabled"` (or `WANDB_MODE=disabled`) for notebooks/CI, or `mode="offline"` to log locally and sync later. Online mode needs `wandb login` (an API key) first.

### Expecting W&B to already be logging

It isn't (§2.2) — no `src/` code calls it. You must wire a callback (§2.4). The dependency being installed doesn't mean it's active.

### Confusing W&B with the results output

W&B is *live telemetry*, not the deliverable — the MATLAB analysis reads `CM_Table.mat`, not W&B (§2.5). Don't drop the `.mat` writer when you add W&B; keep both.

### CI breaks after adding a W&B callback

Guard it: default to `disabled` mode when no API key is present, so CI and offline runs don't fail. A hard `wandb.init(mode="online")` in the training path will break any environment without credentials.

### Metrics not appearing in the dashboard

Check you called `run.log({...})` (per-step metrics) and `run.finish()` (flush). A run that never `finish`es may not sync its last steps; `summary` values need `run.summary[...] = ...` explicitly.

## Section 6 — Further reading

- [Weights & Biases quickstart](https://docs.wandb.ai/quickstart) — `init`/`log`/`finish` and the dashboard.
- [`src/neural_data_decoding/training/lifecycle.py`](../../src/neural_data_decoding/training/lifecycle.py) — the `EpochCallback` hook a logger attaches to.
- [08.3 monitor table compatibility](08.3_monitor_table_compatibility.ipynb) — the callback system W&B plugs into.

Next up: **[08.6 running MATLAB analysis on Python output](08.6_running_matlab_analysis_on_python_output.ipynb)** — closing the loop: train in Python, aggregate and plot in MATLAB.